# DE2 — Lab 1: Structured Streaming Pipeline (10%)
Author : Badr TAJINI - Data Engineering II (Data-Intensive Workloads) - ESIEE 2025-2026
Track: A — Esports
Goal: Build a Structured Streaming pipeline with windowed aggregation, watermarks, and a Parquet sink. Monitor via `query.lastProgress` and the Streaming UI. Deliver a before/after optimization report.

In [1]:
import os
from urllib.parse import urlparse
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
import time, pathlib, json, io, contextlib, shutil, csv
from datetime import datetime

DE2_SPARK_DRIVER_HOST = os.environ.get("DE2_SPARK_DRIVER_HOST", "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)


def show_spark_ui(spark_session):
    ui_url = spark_session.sparkContext.uiWebUrl
    print("Spark version:", spark_session.version)
    if ui_url:
        ui_port = urlparse(ui_url).port or 4040
        print("Spark UI:", ui_url)
        print("Spark UI (WSL/Windows browser):", f"http://localhost:{ui_port}")
    else:
        print("Spark UI: not available")

spark = SparkSession.builder \
    .appName("DE2-Lab1-Streaming") \
    .master("local[*]") \
    .config("spark.driver.host", DE2_SPARK_DRIVER_HOST) \
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.ui.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .getOrCreate()

show_spark_ui(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 00:20:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.1
Spark UI: http://127.0.0.1:4040
Spark UI (WSL/Windows browser): http://localhost:4040


## 1. Define Schema & Prepare Landing Directory
Define an explicit `StructType` schema for Track A (Esports). Create a landing directory where files will be dropped to simulate a streaming source.

In [2]:
# Schema for Track A — Esports match-end events
schema = StructType([
    StructField("match_id",       IntegerType(),   True),
    StructField("match_end_time", TimestampType(), True),
    StructField("team",           StringType(),    True),
    StructField("kills",          IntegerType(),   True),
    StructField("deaths",         IntegerType(),   True),
    StructField("duration_sec",   IntegerType(),   True),
])

# Landing directory where files will arrive one by one
landing_dir = "data/landing/"
source_dir  = "data/source/"
pathlib.Path(landing_dir).mkdir(parents=True, exist_ok=True)
pathlib.Path(source_dir).mkdir(parents=True, exist_ok=True)

# Clean landing dir from any previous run
for f in pathlib.Path(landing_dir).glob("*.json"):
    f.unlink()

print("Landing dir:", landing_dir)
print("Source dir :", source_dir)
print("Source files available:", sorted(p.name for p in pathlib.Path(source_dir).glob("*.json")))

Landing dir: data/landing/
Source dir : data/source/
Source files available: ['batch_01.json', 'batch_02.json', 'batch_03.json', 'batch_04.json', 'batch_05.json', 'batch_06.json', 'batch_07.json', 'batch_08.json', 'batch_09.json', 'batch_10.json']


## 2. Create Streaming Source
Use `spark.readStream` to create a streaming DataFrame from the landing directory.

In [3]:
df_stream = (spark.readStream
    .schema(schema)
    .option("maxFilesPerTrigger", 1)   # process one file per micro-batch
    .json(landing_dir))

print("Is streaming:", df_stream.isStreaming)
df_stream.printSchema()

Is streaming: True
root
 |-- match_id: integer (nullable = true)
 |-- match_end_time: timestamp (nullable = true)
 |-- team: string (nullable = true)
 |-- kills: integer (nullable = true)
 |-- deaths: integer (nullable = true)
 |-- duration_sec: integer (nullable = true)



## 3. Watermark + Windowed Aggregation
Apply a watermark on `match_end_time`, then group by a tumbling window of 1 hour and the team.

In [4]:
windowed = (df_stream
    .withWatermark("match_end_time", "15 minutes")
    .groupBy(
        F.window("match_end_time", "1 hour"),   # tumbling 1h
        F.col("team")
    )
    .agg(
        F.count("*").alias("event_count"),
        F.avg("kills").alias("avg_kills"),
        F.avg("duration_sec").alias("avg_duration_sec")
    ))

print("Windowed schema:")
windowed.printSchema()

# Save the streaming plan as evidence
pathlib.Path("proof").mkdir(parents=True, exist_ok=True)
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    windowed.explain("formatted")
with open("proof/plan_streaming.txt", "w") as f:
    f.write(buf.getvalue())
print("Plan saved to proof/plan_streaming.txt")

Windowed schema:
root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- team: string (nullable = true)
 |-- event_count: long (nullable = false)
 |-- avg_kills: double (nullable = true)
 |-- avg_duration_sec: double (nullable = true)

Plan saved to proof/plan_streaming.txt


## 4. Write to Parquet Sink — Run #1 (baseline: trigger 10s, watermark 15min)
Write the windowed results to a Parquet sink with a trigger interval of 10 seconds. Use `checkpointLocation` for fault tolerance.

In [5]:
# Clean previous run #1 outputs
for path in ["outputs/lab1/stream_sink_v1", "outputs/lab1/checkpoint_v1"]:
    if pathlib.Path(path).exists():
        shutil.rmtree(path)

pathlib.Path("outputs/lab1/stream_sink_v1").mkdir(parents=True, exist_ok=True)
pathlib.Path("outputs/lab1/checkpoint_v1").mkdir(parents=True, exist_ok=True)

query = (windowed.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", "outputs/lab1/stream_sink_v1")
    .option("checkpointLocation", "outputs/lab1/checkpoint_v1")
    .trigger(processingTime="10 seconds")
    .start())

print("Query name:", query.name)
print("Query id  :", query.id)
print("Query is active:", query.isActive)

26/05/10 00:21:02 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Query name: None
Query id  : b728863a-7bc0-44f2-992a-867ced9c5456
Query is active: True


## 4-bis. Feed files one by one (simulate streaming arrivals)
This cell copies JSON files from `data/source/` to `data/landing/` with a 12s delay between each. Spark detects each new file and processes it.

In [6]:
source_files = sorted(pathlib.Path(source_dir).glob("*.json"))
print(f"Found {len(source_files)} source files. Feeding to landing dir...")

for i, src in enumerate(source_files, start=1):
    dst = pathlib.Path(landing_dir) / src.name
    shutil.copy(src, dst)
    print(f"[{i}/{len(source_files)}] dropped {src.name} -> {dst}")
    time.sleep(12)   # leave Spark time to process this micro-batch

print("All files fed. Letting query process the last batch...")
time.sleep(15)

Found 10 source files. Feeding to landing dir...
[1/10] dropped batch_01.json -> data/landing/batch_01.json


[2/10] dropped batch_02.json -> data/landing/batch_02.json


[Stage 5:=======================>                               (85 + 16) / 200]

[3/10] dropped batch_03.json -> data/landing/batch_03.json


[4/10] dropped batch_04.json -> data/landing/batch_04.json


...

[Stage 17:=========================================>           (158 + 16) / 200]

[8/10] dropped batch_08.json -> data/landing/batch_08.json


[9/10] dropped batch_09.json -> data/landing/batch_09.json


[10/10] dropped batch_10.json -> data/landing/batch_10.json


All files fed. Letting query process the last batch...


## 5. Monitor — `query.lastProgress` & save metrics for Run #1

In [7]:
progress_v1 = query.lastProgress
print(json.dumps(progress_v1, indent=2, default=str))

with open("proof/query_progress_v1.json", "w") as f:
    json.dump(progress_v1, f, indent=2, default=str)

# Stop run #1
query.stop()
print("\nQuery v1 stopped. Active:", query.isActive)

{
  "id": "b728863a-7bc0-44f2-992a-867ced9c5456",
  "runId": "b82334d4-962b-49dd-8635-de2df0d72531",
  "name": null,
  "timestamp": "2026-05-09T22:32:30.000Z",
  "batchId": 13,
  "batchDuration": 4,
  "durationMs": {
    "triggerExecution": 4,
    "latestOffset": 4
  },
  "eventTime": {
    "watermark": "2026-05-09T12:41:19.000Z"
  },
  "stateOperators": [
    {
      "operatorName": "stateStoreSave",
      "numRowsTotal": 7,
      "numRowsUpdated": 0,
      "numRowsRemoved": 0,
      "allUpdatesTimeMs": 267,
      "allRemovalsTimeMs": 444,
      "commitTimeMs": 19371,
      "memoryUsedBytes": 91560,
      "numRowsDroppedByWatermark": 0,
      "numShufflePartitions": 200,
      "numStateStoreInstances": 200,
      "customMetrics": {
        "stateOnCurrentVersionSizeBytes": 25736,
        "loadedMapCacheHitCount": 4800,
        "loadedMapCacheMissCount": 0
      }
    }
  ],
  "sources": [
    {
      "description": "FileStreamSource[file:/home/ranajabry/de2/lab1/data/landing]",
      

26/05/10 00:32:37 WARN DAGScheduler: Failed to cancel job group b82334d4-962b-49dd-8635-de2df0d72531. Cannot find active jobs for it.
26/05/10 00:32:37 WARN DAGScheduler: Failed to cancel job group b82334d4-962b-49dd-8635-de2df0d72531. Cannot find active jobs for it.


## 6. Before/After Optimization — Run #2 (trigger 30s, watermark 5min)

In [8]:
# Re-define windowed with shorter watermark
windowed_v2 = (df_stream
    .withWatermark("match_end_time", "5 minutes")
    .groupBy(
        F.window("match_end_time", "1 hour"),
        F.col("team")
    )
    .agg(
        F.count("*").alias("event_count"),
        F.avg("kills").alias("avg_kills"),
        F.avg("duration_sec").alias("avg_duration_sec")
    ))

for path in ["outputs/lab1/stream_sink_v2", "outputs/lab1/checkpoint_v2"]:
    if pathlib.Path(path).exists():
        shutil.rmtree(path)
pathlib.Path("outputs/lab1/stream_sink_v2").mkdir(parents=True, exist_ok=True)
pathlib.Path("outputs/lab1/checkpoint_v2").mkdir(parents=True, exist_ok=True)

# Clean landing dir again and reset feed
for f in pathlib.Path(landing_dir).glob("*.json"):
    f.unlink()

query2 = (windowed_v2.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", "outputs/lab1/stream_sink_v2")
    .option("checkpointLocation", "outputs/lab1/checkpoint_v2")
    .trigger(processingTime="30 seconds")
    .start())

print("Query v2 started. Feeding files again...")
for i, src in enumerate(source_files, start=1):
    dst = pathlib.Path(landing_dir) / src.name
    shutil.copy(src, dst)
    print(f"[{i}/{len(source_files)}] dropped {src.name}")
    time.sleep(15)

time.sleep(35)
progress_v2 = query2.lastProgress
print(json.dumps(progress_v2, indent=2, default=str))

with open("proof/query_progress_v2.json", "w") as f:
    json.dump(progress_v2, f, indent=2, default=str)

query2.stop()
print("\nQuery v2 stopped. Active:", query2.isActive)

26/05/10 00:51:33 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Query v2 started. Feeding files again...
[1/10] dropped batch_01.json


[2/10] dropped batch_02.json


[3/10] dropped batch_03.json
[4/10] dropped batch_04.json


[5/10] dropped batch_05.json
[6/10] dropped batch_06.json


[7/10] dropped batch_07.json
[8/10] dropped batch_08.json


[9/10] dropped batch_09.json
[10/10] dropped batch_10.json


{
  "id": "7c4d6f7b-12d5-4d3b-838a-b7749910b3b0",
  "runId": "92f065e0-3b51-4526-9a31-731d8238ad72",
  "name": null,
  "timestamp": "2026-05-09T22:54:30.000Z",
  "batchId": 6,
  "batchDuration": 1945,
  "durationMs": {
    "triggerExecution": 1945,
    "queryPlanning": 22,
    "getBatch": 6,
    "commitOffsets": 19,
    "latestOffset": 30,
    "addBatch": 1844,
    "walCommit": 18
  },
  "eventTime": {
    "min": "2026-05-09T11:03:11.000Z",
    "avg": "2026-05-09T11:15:08.090Z",
    "watermark": "2026-05-09T10:52:33.000Z",
    "max": "2026-05-09T11:27:34.000Z"
  },
  "stateOperators": [
    {
      "operatorName": "stateStoreSave",
      "numRowsTotal": 13,
      "numRowsUpdated": 6,
      "numRowsRemoved": 0,
      "allUpdatesTimeMs": 411,
      "allRemovalsTimeMs": 577,
      "commitTimeMs": 19125,
      "memoryUsedBytes": 93080,
      "numRowsDroppedByWatermark": 0,
      "numShufflePartitions": 200,
      "numStateStoreInstances": 200,
      "customMetrics": {
        "stateOnCurre

26/05/10 00:54:38 WARN DAGScheduler: Failed to cancel job group 92f065e0-3b51-4526-9a31-731d8238ad72. Cannot find active jobs for it.
26/05/10 00:54:38 WARN DAGScheduler: Failed to cancel job group 92f065e0-3b51-4526-9a31-731d8238ad72. Cannot find active jobs for it.


## 7. Build `lab1_metrics_log.csv` from both runs

In [9]:
def extract_metrics(progress, run_id, task, trigger_interval, watermark_duration):
    if not progress:
        return [run_id, task, trigger_interval, watermark_duration, 0, 0.0, 0, 0, 0, datetime.now().isoformat(), "no progress captured"]
    p = progress
    state = (p.get("stateOperators") or [{}])[0]
    return [
        run_id,
        task,
        trigger_interval,
        watermark_duration,
        p.get("numInputRows", 0),
        round(float(p.get("processedRowsPerSecond") or 0), 2),
        (p.get("durationMs") or {}).get("triggerExecution", 0),
        state.get("numRowsTotal", 0),
        state.get("numRowsUpdated", 0),
        p.get("timestamp", datetime.now().isoformat()),
        ""
    ]

rows = [
    extract_metrics(progress_v1, 1, "baseline_trigger_10s_wm_15min", "10s", "15 minutes"),
    extract_metrics(progress_v2, 2, "optimized_trigger_30s_wm_5min", "30s", "5 minutes"),
]

header = ["run_id", "task", "trigger_interval", "watermark_duration",
          "numInputRows", "processedRowsPerSecond", "triggerExecution_ms",
          "stateRowsTotal", "stateRowsUpdated", "timestamp", "notes"]

with open("lab1_metrics_log.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(header)
    w.writerows(rows)

print("lab1_metrics_log.csv written:")
for r in rows:
    print(r)

lab1_metrics_log.csv written:
[1, 'baseline_trigger_10s_wm_15min', '10s', '15 minutes', 0, 0.0, 4, 7, 0, '2026-05-09T22:32:30.000Z', '']
[2, 'optimized_trigger_30s_wm_5min', '30s', '5 minutes', 11, 5.66, 1945, 13, 6, '2026-05-09T22:54:30.000Z', '']


## 8. Read back the Parquet sink to verify output

In [10]:
result_v1 = spark.read.parquet("outputs/lab1/stream_sink_v1")
print("Run #1 — rows in Parquet sink:", result_v1.count())
result_v1.show(20, truncate=False)

result_v2 = spark.read.parquet("outputs/lab1/stream_sink_v2")
print("Run #2 — rows in Parquet sink:", result_v2.count())
result_v2.show(20, truncate=False)

Run #1 — rows in Parquet sink: 27
+------------------------------------------+-----+-----------+------------------+------------------+
|window                                    |team |event_count|avg_kills         |avg_duration_sec  |
+------------------------------------------+-----+-----------+------------------+------------------+
|{2026-05-09 11:00:00, 2026-05-09 12:00:00}|Gen.G|2          |29.5              |2968.5            |
|{2026-05-09 11:00:00, 2026-05-09 12:00:00}|G2   |1          |7.0               |2680.0            |
|{2026-05-09 12:00:00, 2026-05-09 13:00:00}|Gen.G|2          |19.0              |2994.0            |
|{2026-05-09 13:00:00, 2026-05-09 14:00:00}|Gen.G|2          |3.5               |1545.5            |
|{2026-05-09 11:00:00, 2026-05-09 12:00:00}|DRX  |4          |21.0              |2239.75           |
|{2026-05-09 11:00:00, 2026-05-09 12:00:00}|TSM  |4          |15.5              |2746.5            |
|{2026-05-09 12:00:00, 2026-05-09 13:00:00}|FNC  |3      

In [11]:
spark.stop()
print("Lab 1 complete.")

Lab 1 complete.


26/05/10 01:08:52 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:971)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:945)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:746)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:572)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:358)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1144)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:642)
	at java.base/java.lang.Thread.